# Climate Claim Extraction mit ClimateBERT

Dieses Notebook extrahiert in Nachhaltigkeitsberichten potentielle Green Claims und speichert sie zur weiteren Auswertung in einer CSV-Datei. Als Modell wird [`climatebert/environmental-claims`](https://huggingface.co/climatebert/environmental-claims) verwendet.

## 1. Voraussetzungen

Stelle sicher, dass die Python-Umgebung die in `requirements.txt` aufgeführten Pakete enthält. Für dieses Notebook werden insbesondere `transformers`, `torch`, `pandas` und `pdfplumber` benötigt.

In [ ]:
from pathlib import Path
import re
from typing import Dict, List

import pandas as pd
import pdfplumber
from transformers import AutoModelForSequenceClassification, AutoTokenizer, TextClassificationPipeline

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)

ESG_DIR = Path("ESG Reports")
PDF_PATHS = sorted(ESG_DIR.glob("*.pdf"))
PDF_PATHS

## 2. Hilfsfunktionen

Die folgenden Funktionen lesen PDF-Dateien ein, normalisieren den Text und zerlegen ihn in Sätze. Einfache Heuristiken (z. B. nach Punkten, Frage- und Ausrufezeichen) werden verwendet, um die Sätze zu bestimmen. Für wissenschaftliche Analysen empfiehlt sich, diese Vorverarbeitung bei Bedarf zu verfeinern.

In [ ]:
sentence_split_regex = re.compile(r"(?<=[.!?])\s+(?=[A-ZÄÖÜ])")

def extract_sentences_from_pdf(path: Path) -> List[Dict[str, str]]:
    sentences = []
    with pdfplumber.open(path) as pdf:
        for page_number, page in enumerate(pdf.pages, start=1):
            text = page.extract_text() or ""
            text = re.sub(r"\s+", " ", text).strip()
            if not text:
                continue
            for sentence in sentence_split_regex.split(text):
                sentence = sentence.strip()
                if len(sentence) < 25:
                    continue
                sentences.append({
                    "document_id": path.name,
                    "page": page_number,
                    "sentence": sentence
                })
    return sentences

sample_sentences = extract_sentences_from_pdf(PDF_PATHS[0]) if PDF_PATHS else []
len(sample_sentences), sample_sentences[:3]

## 3. Claim-Klassifikation mit ClimateBERT

Das Modell liefert eine binäre Klassifikation (Claim vs. Non-Claim). Wir setzen eine Score-Schwelle von 0,5; alle Sätze mit höherem Score gelten als Claims. Die Schwelle kann je nach gewünschter Präzision/Recall angepasst werden.

In [ ]:
model_name = "climatebert/environmental-claims"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name)
claim_pipeline = TextClassificationPipeline(model=model, tokenizer=tokenizer, return_all_scores=False, device=-1)

threshold = 0.5
records = []
for path in PDF_PATHS:
    for sentence_entry in extract_sentences_from_pdf(path):
        prediction = claim_pipeline(
            sentence_entry["sentence"],
            truncation=True,
            max_length=tokenizer.model_max_length
        )
        label = prediction[0]["label"]
        score = float(prediction[0]["score"])
        is_claim = int(label.lower().endswith("claim")) if "claim" in label.lower() else int(score >= threshold)
        if score >= threshold:
            records.append({
                **sentence_entry,
                "label": label,
                "score": score,
                "is_claim": is_claim
            })

claims_df = pd.DataFrame.from_records(records)
claims_df.head()

## 4. Speicherung der Ergebnisse

Die extrahierten Claims werden in einer CSV-Datei (`data/claims.csv`) abgelegt. Diese Datei dient als Eingabe für die KPI-Extraktion sowie das nachgelagerte LLM-Evaluationsnotebook.

In [ ]:
claims_csv_path = DATA_DIR / "claims.csv"
claims_df.to_csv(claims_csv_path, index=False)
claims_csv_path